# India Home Remedies SLM

Fine-tuning TinyLlama-1.1B-Chat on traditional Indian home remedies.

---

> **Medical Disclaimer**: This model provides traditional Indian home remedy suggestions only. It is NOT a medical diagnosis tool and should NOT replace professional medical advice. Always consult a licensed physician for serious or persistent health conditions.

---

| Component | Choice |
|---|---|
| Base Model | TinyLlama/TinyLlama-1.1B-Chat-v1.0 (1.1B params) |
| Fine-tuning | LoRA r=16 via peft + trl SFTTrainer |
| Quantisation | 4-bit NF4 via bitsandbytes (GPU) |
| Package manager | uv |
| Experiment tracking | wandb |

Run all cells top-to-bottom.


In [ ]:
# Cell 1 - Install dependencies
# !! IMPORTANT: After running this cell, go to Kernel -> Restart Kernel, then continue from Cell 2 !!
# Pinned versions are tested to be compatible with each other.
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *args])

# PyTorch with CUDA 11.8 support
pip('torch','torchvision','torchaudio',
    '--index-url','https://download.pytorch.org/whl/cu118')

# Core ML stack - pinned to known-compatible versions
# transformers 4.41.2 + peft 0.11.1 + trl 0.8.6 are tested together
pip(
    'transformers==4.41.2',
    'peft==0.11.1',
    'trl==0.8.6',
    'datasets>=2.19.0',
    'bitsandbytes>=0.43.1',
    'accelerate>=0.30.0',
    'scipy',
    'sentencepiece',
    'wandb',
    'gradio>=4.0.0',
    'fastapi',
    'uvicorn',
    'python-dotenv',
    'evaluate',
    'rouge_score',
    'nltk',
)

print('✅ All packages installed.')
print('⚠️  RESTART THE KERNEL NOW before running Cell 2!')
print('   Kernel -> Restart Kernel  (or Ctrl+Shift+P -> Restart Kernel)')


In [ ]:
# Cell 2 - Imports and GPU check
import os, json, re, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_GPU = device == 'cuda'
print(f'Device : {device}')
if USE_GPU:
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU found - training will be very slow.')

ROOT       = Path('.').resolve()
DATA_RAW   = ROOT / 'data' / 'raw'
DATA_PROC  = ROOT / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'outputs' / 'remedies-slm'
for d in [DATA_RAW, DATA_PROC, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f'Directories ready under {ROOT}')


In [ ]:
# Cell 3 - Weights and Biases setup
# Set WANDB_API_KEY in a .env file at project root, or paste key when prompted
import wandb
from dotenv import load_dotenv
load_dotenv()
wandb.login()
WANDB_PROJECT  = 'india-home-remedies-slm'
WANDB_RUN_NAME = 'tinyllama-lora-v1'
print(f'W&B ready | project={WANDB_PROJECT} | run={WANDB_RUN_NAME}')


In [ ]:
# Cell 4 - Seed dataset (110 Q&A pairs)
# Categories: Cold/Cough, Digestion, Fever, Headache, Skin, Joints, Sleep, Teeth
import json
from pathlib import Path
ROOT     = Path('.').resolve()
DATA_RAW = ROOT / 'data' / 'raw'
DATA_RAW.mkdir(parents=True, exist_ok=True)

SEED_DATA = [
  {
    "instruction": "What home remedy helps with a cold and blocked nose?",
    "input": "",
    "output": "Drink warm tulsi-ginger tea with honey three times a day. Inhale steam with 2-3 drops of eucalyptus oil or a pinch of ajwain for 10 minutes. Keep yourself well hydrated."
  },
  {
    "instruction": "My throat is sore and I am coughing a lot. Any Indian home remedy?",
    "input": "",
    "output": "Mix 1 tsp honey with a pinch of turmeric and consume twice daily. Gargle with warm salt water every 4-6 hours. Drinking haldi-doodh (turmeric milk) at bedtime helps soothe the throat."
  },
  {
    "instruction": "Suggest a remedy for dry cough.",
    "input": "",
    "output": "Roast a piece of raw ginger and chew it slowly. Mix mulethi (licorice root) powder with honey and lick slowly. Peppercorn tea with tulsi leaves is another effective option."
  },
  {
    "instruction": "How to treat a runny nose naturally?",
    "input": "",
    "output": "Inhale steam with a few drops of eucalyptus oil. Drink warm water with ginger juice and a pinch of black pepper. Keeping yourself warm and avoiding cold foods helps."
  },
  {
    "instruction": "What can I use for nasal congestion at home?",
    "input": "",
    "output": "Prepare a saline nasal rinse with 1/4 tsp salt in 1 cup of warm water. Use peppermint or eucalyptus steam inhalation. Eating raw garlic or adding it to soup also helps."
  },
  {
    "instruction": "My child has a mild cold. What natural remedy is safe?",
    "input": "",
    "output": "For children above 2 years, give warm tulsi-honey tea. Applying warm coconut oil mixed with camphor on the chest provides soothing relief. Keep the child warm and well hydrated."
  },
  {
    "instruction": "What helps with a wet cough and phlegm?",
    "input": "",
    "output": "Boil water with ginger, peppercorns, cinnamon, and cloves. Strain and drink with honey. Eating radish with raw honey breaks down phlegm. Steam inhalation with tulsi leaves is effective."
  },
  {
    "instruction": "How can I reduce sneezing from a cold quickly?",
    "input": "",
    "output": "Smell a freshly cut piece of ginger or prepare ginger tea with black pepper. Inhaling steam helps reduce sneezing. Drinking lukewarm water with a pinch of carom seeds (ajwain) is a traditional remedy."
  },
  {
    "instruction": "What Indian spices help fight a cold?",
    "input": "",
    "output": "Turmeric, fresh ginger, black pepper, tulsi, cloves, and cinnamon are the cornerstone spices. A kadha made by boiling these together with jaggery is a classic Ayurvedic preparation."
  },
  {
    "instruction": "What is kadha and how do I make it for a cold?",
    "input": "",
    "output": "Kadha is a traditional herbal decoction. Boil 2 cups water with 4-5 tulsi leaves, 1-inch ginger, 4 black peppercorns, 2 cloves, and cinnamon for 10 minutes. Strain, add jaggery or honey, and drink warm twice a day."
  },
  {
    "instruction": "Remedy for cough at night that disturbs sleep?",
    "input": "",
    "output": "Drink 1 glass of warm haldi-doodh before sleeping. Elevate your head slightly. Sucking on a mulethi stick before sleeping calms night-time cough."
  },
  {
    "instruction": "How to stop a persistent cough naturally?",
    "input": "",
    "output": "Make a paste of ginger juice, honey, and a pinch of black pepper. Take 1 tsp thrice a day. Drinking warm water with crushed tulsi leaves every few hours also helps. Avoid cold drinks and bananas."
  },
  {
    "instruction": "What drink helps with cold and fever together?",
    "input": "",
    "output": "Boil water with tulsi, ginger, cloves, and black pepper to make a kadha. Add lemon juice and honey. This helps reduce fever, soothe cold symptoms, and boost immunity."
  },
  {
    "instruction": "How to get relief from blocked ears during cold?",
    "input": "",
    "output": "Try steam inhalation to open the Eustachian tubes. Chewing gum or yawning equalizes ear pressure. A warm cloth over the ear provides soothing relief. See a doctor if symptoms persist beyond 5 days."
  },
  {
    "instruction": "What food should I eat if I have a cold?",
    "input": "",
    "output": "Eat light, warm foods like khichdi, moong dal soup, or vegetable broth. Avoid cold, heavy, or fried foods. Garlic, ginger, and turmeric boost immunity. Avoid dairy if you have excessive phlegm."
  },
  {
    "instruction": "Suggest a home remedy for hoarse voice due to cold.",
    "input": "",
    "output": "Gargle with warm salt water twice a day. Drink ginger tea with honey. Rest your voice. Sucking on a piece of licorice (mulethi) or a clove soothes inflamed vocal cords."
  },
  {
    "instruction": "How many days does a cold last and how can I recover faster?",
    "input": "",
    "output": "A common cold typically lasts 5-10 days. Recovery is faster with adequate rest, warm liquids, steam inhalation, and daily kadha. Eating an amla daily boosts vitamin C and speeds recovery."
  },
  {
    "instruction": "Is it safe to eat fruits during a cold?",
    "input": "",
    "output": "Yes. Warm fruits like cooked apple with cinnamon are beneficial. Amla, papaya, and pomegranate are excellent immunity boosters. Avoid cold citrus juices from the fridge."
  },
  {
    "instruction": "How do I use ajwain for cold relief?",
    "input": "",
    "output": "Wrap dry-roasted ajwain (carom seeds) in a thin cloth and inhale the aroma to relieve nasal congestion. You can also boil 1 tsp ajwain in water, strain, and drink the warm water."
  },
  {
    "instruction": "What is the best remedy for cold in cold weather?",
    "input": "",
    "output": "Drink warm sesame tea made by boiling 1 tsp sesame seeds in 2 cups water with jaggery and ginger. A warm mustard oil body massage alleviates chills and body aches."
  },
  {
    "instruction": "I feel bloated after eating. What can I take at home?",
    "input": "",
    "output": "Chew 1 tsp of roasted ajwain with a pinch of black salt and drink warm water. Fennel seeds (saunf) after meals also relieve bloating. Avoid carbonated drinks and heavy fried foods."
  },
  {
    "instruction": "I have acidity after eating spicy food. What helps?",
    "input": "",
    "output": "Drink a glass of cold milk or chew fennel seeds (saunf) after meals. Coconut water is very effective. Try jeera water - boil 1 tsp cumin in 2 cups water, cool, and sip slowly."
  },
  {
    "instruction": "What is a good home remedy for constipation?",
    "input": "",
    "output": "Drink 1-2 glasses of warm water first thing in the morning. Having 1 tsp of isabgol (psyllium husk) in warm milk before bed is very effective. Soaked raisins or figs overnight also help."
  },
  {
    "instruction": "What helps with an upset stomach and diarrhea?",
    "input": "",
    "output": "Eat the BRAT diet - bananas, rice, applesauce, and toast. Drink ORS or homemade nimbu-pani with a pinch of salt and sugar. Boiled rice water with a pinch of salt soothes an upset stomach."
  },
  {
    "instruction": "I have gas and stomach pain. Any home remedy?",
    "input": "",
    "output": "Boil jeera, ajwain, and saunf together in water, strain, and drink warm. Asafoetida (hing) dissolved in warm water relieves gas quickly. Gently massaging the abdomen clockwise also helps."
  },
  {
    "instruction": "What is a good remedy for indigestion?",
    "input": "",
    "output": "Drink ginger tea before or after meals. Mix 1 tsp apple cider vinegar in warm water and drink before meals. Eating a small piece of raw ginger with rock salt before a meal stimulates digestive enzymes."
  },
  {
    "instruction": "How to treat nausea naturally?",
    "input": "",
    "output": "Sip on cold ginger tea or dry ginger lemonade. Smell a halved lemon or inhale peppermint oil. Eating a few dry crackers settles the stomach. Avoid lying down immediately after a meal."
  },
  {
    "instruction": "How to relieve heartburn quickly at home?",
    "input": "",
    "output": "Chew a piece of clove or cardamom slowly. Drink a glass of cold milk. Mix 1 tsp baking soda in half a glass of water for quick relief (not more than twice a week). Sleeping on your left side helps."
  },
  {
    "instruction": "What helps with slow digestion after heavy meals?",
    "input": "",
    "output": "Drink warm jeera water after meals. Eating a slice of papaya or pineapple provides natural digestive enzymes. A 10-15 minute gentle walk after eating significantly improves digestion."
  },
  {
    "instruction": "How to stop frequent loose motions naturally?",
    "input": "",
    "output": "Eat 1-2 bananas or plain rice with curd. Drink buttermilk with a pinch of salt and roasted jeera. Boiling raw guava leaves and drinking the water helps firm stools."
  },
  {
    "instruction": "What can I eat when I have a stomach infection?",
    "input": "",
    "output": "Stick to plain rice, boiled dal, and curd. Drink plenty of fluids - ORS, coconut water. Avoid raw vegetables and meat. Jeera rice with plain dal is an ideal meal."
  },
  {
    "instruction": "How to cure hiccups with a home remedy?",
    "input": "",
    "output": "Hold your breath for 10-15 seconds then exhale slowly. Eat a teaspoon of sugar. Sip cold water slowly. Sucking on a lemon slice with salt stops hiccups quickly."
  },
  {
    "instruction": "What food remedies help with irritable bowel syndrome (IBS)?",
    "input": "",
    "output": "Drink fennel seed tea (boil 1 tsp saunf in 2 cups water). Avoid raw onions and spicy food. Eating curd with isabgol daily regulates bowel movement."
  },
  {
    "instruction": "How to reduce excessive burping at home?",
    "input": "",
    "output": "Eat slowly and chew food thoroughly. Drinking ginger tea with honey after meals reduces excessive gas. Chewing fennel seeds or cardamom after meals is a classic Indian digestive aid."
  },
  {
    "instruction": "What are natural antacids available in an Indian kitchen?",
    "input": "",
    "output": "Cold milk, coconut water, jaggery, fennel seeds, and coriander seed water are natural antacids. Amla juice on an empty stomach reduces hyperacidity."
  },
  {
    "instruction": "I have pain around my navel. What home remedy helps?",
    "input": "",
    "output": "Massage warm castor oil or sesame oil clockwise around the navel. Drinking warm ajwain water relieves cramps and gas. If pain is severe or persistent, seek medical attention."
  },
  {
    "instruction": "How to get rid of stomach worms with home remedies?",
    "input": "",
    "output": "Eat raw papaya seeds (1 tsp) with honey for 3-7 days. Raw garlic on an empty stomach is a traditional deworming remedy. Consult a doctor if symptoms are severe."
  },
  {
    "instruction": "What drink helps improve gut health naturally?",
    "input": "",
    "output": "Drink chaas (buttermilk) with roasted jeera powder daily after lunch. Kefir or homemade lassi is probiotic-rich. Amla juice or triphala water on an empty stomach supports gut detoxification."
  },
  {
    "instruction": "How to treat vomiting at home?",
    "input": "",
    "output": "Sip cold ginger tea slowly. Place a cold wet cloth on your forehead and rest. Eat plain dry toast. Avoid solid food for 1-2 hours after vomiting. Gradual rehydration with ORS is key."
  },
  {
    "instruction": "How to reduce stomach heat or pitta imbalance?",
    "input": "",
    "output": "Drink chilled coconut water, rose water, or coriander seed water. Avoid spicy, fried, sour foods. Eating amla, cucumber, and mint-based foods cools excess pitta."
  },
  {
    "instruction": "I have mild fever. What home remedies help?",
    "input": "",
    "output": "Drink tulsi decoction (boil 5 tulsi leaves and 1 tsp ginger in 2 cups water), strain, and add honey. Place a cool damp cloth on your forehead. Stay hydrated with coconut water and herbal teas."
  },
  {
    "instruction": "How to reduce body temperature naturally?",
    "input": "",
    "output": "Apply a damp cloth soaked in room-temperature water on the forehead, armpits, and wrists. Drink cold water frequently. Apple cider vinegar compresses on the body draw heat out."
  },
  {
    "instruction": "What is a natural immunity booster using kitchen ingredients?",
    "input": "",
    "output": "Drink warm water with 1 tsp turmeric, a pinch of black pepper, and 1 tsp honey every morning. Eat 1-2 cloves of raw garlic daily. Amla juice provides a powerful daily dose of vitamin C."
  },
  {
    "instruction": "How to treat low-grade fever in a child at home?",
    "input": "",
    "output": "Give the child plenty of fluids and apply a cold compress to the forehead. Rub with a sponge soaked in lukewarm water. Consult a doctor if fever exceeds 38.5 degrees Celsius or lasts more than 2 days."
  },
  {
    "instruction": "What helps with body ache and fever?",
    "input": "",
    "output": "Drink a warm kadha of tulsi, ginger, black pepper, and cloves. Take warm salt-water baths to soothe aching muscles. Stay well hydrated and rest completely."
  },
  {
    "instruction": "How to boost immunity during seasonal change?",
    "input": "",
    "output": "Drink chyawanprash (1 tsp) with warm milk daily. Eat soaked almonds, walnuts, and raisins every morning. Include turmeric, ginger, garlic, and tulsi in daily cooking."
  },
  {
    "instruction": "What are the best herbs for fighting infection naturally?",
    "input": "",
    "output": "Tulsi, neem, giloy (Guduchi), turmeric, and ginger are powerful antimicrobial herbs. Giloy juice or capsules are widely used in Ayurveda to fight infections and boost immunity."
  },
  {
    "instruction": "How to make giloy kadha for immunity?",
    "input": "",
    "output": "Boil 1 inch of giloy stem (or 1 tsp giloy powder) in 2 cups of water for 10 minutes until reduced to 1 cup. Add black pepper and honey. Drink warm once daily on an empty stomach."
  },
  {
    "instruction": "What should I eat during a viral fever?",
    "input": "",
    "output": "Eat light, easily digestible food: khichdi, moong dal soup, vegetable broth. Avoid heavy fried foods and meat. Drink plenty of warm fluids. Drumstick leaves soup is very nutritious."
  },
  {
    "instruction": "How to recover faster after a fever breaks?",
    "input": "",
    "output": "Continue drinking fluids and eating light foods for 2-3 days. Drink coconut water to replenish electrolytes. Ashwagandha powder in warm milk restores strength and energy levels."
  },
  {
    "instruction": "What causes recurring fever and what home steps can I take?",
    "input": "",
    "output": "Recurring fever often indicates an underlying infection requiring doctor evaluation. Stay hydrated and consume immunity-boosting herbs like tulsi, giloy, and neem."
  },
  {
    "instruction": "How to prevent viral infections during monsoon season?",
    "input": "",
    "output": "Drink boiled water and avoid street food. Add tulsi, ginger, and turmeric to daily meals. Keep surroundings dry. Take 1 tsp chyawanprash daily. Wash hands frequently."
  },
  {
    "instruction": "Is there an Indian remedy for supporting dengue recovery?",
    "input": "",
    "output": "Papaya leaf juice is traditionally used as a supportive remedy to help increase platelet count in dengue. Neem and giloy are used in Ayurveda. Dengue is serious - seek immediate medical care."
  },
  {
    "instruction": "How to reduce fatigue and weakness after illness?",
    "input": "",
    "output": "Drink ashwagandha milk - 1 tsp ashwagandha powder in warm milk with honey. Eat protein-rich foods like dal, paneer, or eggs. Soak 5 almonds overnight and eat in the morning."
  },
  {
    "instruction": "What helps with chills during a fever?",
    "input": "",
    "output": "Wrap the body in warm blankets and drink hot herbal teas with tulsi, ginger, and cinnamon. Prepare a warm bath with eucalyptus oil. Hot soup with garlic and ginger warms the body from inside."
  },
  {
    "instruction": "What home remedy helps with a headache?",
    "input": "",
    "output": "Apply a cold or warm compress to the forehead or the back of the neck. Massage peppermint oil on the temples. Drink ginger tea - ginger has natural anti-inflammatory properties."
  },
  {
    "instruction": "How to get rid of a migraine at home?",
    "input": "",
    "output": "Rest in a dark, quiet room. Apply a cold pack to the forehead. Drink strong ginger tea. Massaging lavender or peppermint oil on the temples may reduce migraine intensity."
  },
  {
    "instruction": "What causes headaches and how can I prevent them naturally?",
    "input": "",
    "output": "Common causes include dehydration, eye strain, stress, and poor posture. Drink 8-10 glasses of water daily. Practice deep breathing or yoga. Maintain a regular sleep schedule."
  },
  {
    "instruction": "What Indian spice helps with headaches?",
    "input": "",
    "output": "Ginger is highly effective - drink ginger tea or apply ginger paste on the forehead. Clove oil applied to temples provides relief. A paste of sandalwood powder and rose water on the forehead cools and soothes."
  },
  {
    "instruction": "How to relieve tension headache at home?",
    "input": "",
    "output": "Gently massage the scalp, neck, and shoulder muscles. Apply warm compress to the neck area. Practice deep breathing. Peppermint tea or peppermint oil on the temples are classic tension headache remedies."
  },
  {
    "instruction": "What helps with headache due to heat or sun?",
    "input": "",
    "output": "Move to a cool, shaded area immediately. Apply a cold damp cloth to the forehead and neck. Drink cold coconut water or buttermilk to restore electrolytes."
  },
  {
    "instruction": "How to stop a headache without medicine?",
    "input": "",
    "output": "Try acupressure - press the webbing between your thumb and index finger firmly for 4-5 minutes. Apply peppermint oil on the temples. Drink a glass of water, as dehydration is a common trigger."
  },
  {
    "instruction": "What relieves sinus headache naturally?",
    "input": "",
    "output": "Steam inhalation with eucalyptus or peppermint oil relieves sinus pressure. Apply a warm compress over the nose and forehead. Drinking warm ginger-turmeric tea reduces inflammation."
  },
  {
    "instruction": "How to reduce headaches from screen time?",
    "input": "",
    "output": "Follow the 20-20-20 rule: every 20 minutes, look at something 20 feet away for 20 seconds. Apply rose water-soaked cotton pads on closed eyes. Take short breaks every hour."
  },
  {
    "instruction": "What foods help with chronic headaches?",
    "input": "",
    "output": "Stay hydrated and eat magnesium-rich foods: leafy greens, nuts, bananas, and seeds. Reduce processed food and aged cheeses which are migraine triggers. Eat small, regular meals."
  },
  {
    "instruction": "How to relieve headache in pregnancy naturally?",
    "input": "",
    "output": "Rest in a dark, quiet room. Apply a cold compress to the forehead. Drink plenty of water. Always consult your doctor before taking any remedies while pregnant."
  },
  {
    "instruction": "What aromatherapy helps with headaches?",
    "input": "",
    "output": "Lavender and peppermint essential oils are effective. Apply 2 drops of diluted peppermint oil to the temples. Inhaling lavender oil from a diffuser relaxes tension and reduces headache intensity."
  },
  {
    "instruction": "How to use cloves for headache relief?",
    "input": "",
    "output": "Crush 4-5 cloves and place in a thin cloth sachet. Inhale deeply - eugenol in cloves provides natural pain relief. You can also mix clove powder with coconut oil and apply gently on the forehead."
  },
  {
    "instruction": "Does cinnamon help with headaches?",
    "input": "",
    "output": "Yes. Make a thick paste of cinnamon powder with water and apply on the forehead. Let it sit for 30 minutes, then wash off. Cinnamon has anti-inflammatory properties."
  },
  {
    "instruction": "What yoga poses help relieve headaches?",
    "input": "",
    "output": "Child Pose, Legs-up-the-Wall, and Savasana are excellent for headache relief. Alternate nostril breathing (Anulom Vilom) is proven to reduce stress headaches."
  },
  {
    "instruction": "What helps with acne or pimples using home remedies?",
    "input": "",
    "output": "Apply a paste of neem leaves ground with rose water on affected areas. Dab tea tree oil on pimples. Raw honey applied directly is anti-bacterial. Wash your face twice daily."
  },
  {
    "instruction": "How to get glowing skin naturally?",
    "input": "",
    "output": "Apply a face pack of besan (gram flour), turmeric, and rose water weekly. Drink plenty of water. Aloe vera gel applied at night is an excellent hydrator. Eating amla and oranges helps achieve naturally glowing skin."
  },
  {
    "instruction": "What helps with dry, itchy skin in winter?",
    "input": "",
    "output": "Apply warm coconut oil or sesame oil all over the body after a warm bath. Mix glycerin with rose water and apply on the skin. Use aloe vera gel as a natural moisturizer. Avoid hot showers."
  },
  {
    "instruction": "How to treat sunburn at home?",
    "input": "",
    "output": "Apply fresh aloe vera gel directly on sunburned skin. Cool the skin with a cloth soaked in chilled milk. Apply cucumber slices on the affected area. Stay out of the sun until fully healed."
  },
  {
    "instruction": "What home remedy reduces dark circles under the eyes?",
    "input": "",
    "output": "Place cool used green tea bags or cucumber slices on closed eyes for 15 minutes. Mix potato juice with lemon juice and apply under the eye area. Getting 7-8 hours of sleep is the most effective remedy."
  },
  {
    "instruction": "How to treat skin rashes or eczema naturally?",
    "input": "",
    "output": "Apply cold aloe vera gel to soothe inflammation. Oatmeal baths are effective - add 1 cup of finely ground oatmeal to a lukewarm bath. Apply neem paste to infected areas."
  },
  {
    "instruction": "How to remove tan from skin at home?",
    "input": "",
    "output": "Apply a pack of gram flour (besan), turmeric, and lemon juice for 20 minutes. Raw potato juice gently applied on tanned areas lightens over time."
  },
  {
    "instruction": "What helps with oily skin and open pores?",
    "input": "",
    "output": "Apply a face mask of multani mitti (Fuller earth) mixed with rose water weekly. Use chilled rose water as a toner. Aloe vera gel balances oil production without clogging pores."
  },
  {
    "instruction": "How to treat minor skin burns at home?",
    "input": "",
    "output": "Immediately cool the burn under cold running water for 10-15 minutes. Apply pure aloe vera gel generously. Do NOT apply toothpaste or butter on burns. Seek medical help for severe burns."
  },
  {
    "instruction": "What Indian herbs help with skin infections?",
    "input": "",
    "output": "Neem is the most powerful antifungal and antibacterial herb. Apply neem leaf paste on affected areas. Turmeric paste mixed with coconut oil treats ringworm and fungal infections."
  },
  {
    "instruction": "What helps with knee pain at home?",
    "input": "",
    "output": "Apply a warm compress or heating pad on the affected knee for 15 minutes. Massage with warm sesame oil mixed with camphor. Turmeric milk reduces joint inflammation."
  },
  {
    "instruction": "What is a good remedy for muscle soreness after exercise?",
    "input": "",
    "output": "Apply a mustard oil massage with camphor on sore muscles. Soak in an Epsom salt warm bath for 20 minutes. Drink ginger tea to reduce exercise-induced inflammation."
  },
  {
    "instruction": "How to relieve back pain with home remedies?",
    "input": "",
    "output": "Apply a warm compress to the lower back for 15-20 minutes. Massage with eucalyptus oil. Practice Cat-Cow yoga pose. A firm mattress and good posture prevent chronic back pain."
  },
  {
    "instruction": "What helps with arthritis pain in Indian traditional medicine?",
    "input": "",
    "output": "Drink turmeric milk daily - curcumin is a powerful anti-inflammatory. Apply a warm paste of ginger and sesame oil on joints. Castor oil massage before a warm bath is an Ayurvedic remedy."
  },
  {
    "instruction": "How to treat sprained ankle at home?",
    "input": "",
    "output": "Follow RICE method: Rest, Ice (cold pack for 20 min every 2 hours), Compression, and Elevation. Massage gently with warm sesame oil after 48 hours."
  },
  {
    "instruction": "What helps with neck pain and stiffness?",
    "input": "",
    "output": "Apply a warm compress to the neck for 15 minutes. Massage with warm mustard or eucalyptus oil. Gentle neck rotations help restore flexibility. Check your pillow height."
  },
  {
    "instruction": "How to reduce swelling in joints naturally?",
    "input": "",
    "output": "Apply a paste of turmeric and castor oil on swollen joints. Ice packs reduce acute swelling. Ginger tea drunk twice daily reduces chronic joint inflammation."
  },
  {
    "instruction": "What is the best oil for body massage to relieve pain?",
    "input": "",
    "output": "Warm sesame oil (til ka tel) is most widely recommended in Ayurveda for joint lubrication. Mustard oil with camphor is excellent for muscle pain. Mahanarayan oil is an Ayurvedic blend for joint pain."
  },
  {
    "instruction": "How to reduce foot pain from standing all day?",
    "input": "",
    "output": "Soak feet in warm water with Epsom salt and peppermint oil for 20 minutes. Roll your foot over a frozen water bottle. Foot massage with warm sesame oil before bedtime provides great relief."
  },
  {
    "instruction": "What helps with growing pains in children?",
    "input": "",
    "output": "Gently massage the child legs with warm sesame or coconut oil before bedtime. Apply a warm compress on aching areas. A warm bath before sleep helps. Consult a pediatrician if pain is severe."
  },
  {
    "instruction": "What helps me sleep better naturally?",
    "input": "",
    "output": "Drink warm haldi-doodh (turmeric milk) with a pinch of nutmeg before bedtime. Practice deep breathing or Shavasana. Keep a consistent sleep schedule. Brahmi supplement can help with chronic sleep issues."
  },
  {
    "instruction": "How to reduce stress and anxiety naturally?",
    "input": "",
    "output": "Practice alternate nostril breathing (Anulom Vilom) for 10 minutes daily. Brew ashwagandha root tea. Practice yoga or take a daily 30-minute walk. Reducing caffeine after 2 PM reduces anxiety."
  },
  {
    "instruction": "What Indian herbs help with stress relief?",
    "input": "",
    "output": "Ashwagandha is the most powerful adaptogen for stress. Brahmi improves cognitive function and reduces anxiety. Shankhpushpi is an Ayurvedic nerve tonic. Chamomile and tulsi teas are gentler daily options."
  },
  {
    "instruction": "How to calm anxiety before an important event?",
    "input": "",
    "output": "Practice box breathing: inhale 4 counts, hold 4, exhale 4, hold 4. Drink chamomile or tulsi tea. Apply lavender oil on the wrists. A short walk in natural light reduces cortisol rapidly."
  },
  {
    "instruction": "What is a good bedtime drink for better sleep?",
    "input": "",
    "output": "Warm milk with a pinch of nutmeg, turmeric, and a tsp of ashwagandha powder is an excellent sleep aid. Chamomile tea is a popular calming drink. Warm water with honey also promotes sleepiness."
  },
  {
    "instruction": "How to deal with insomnia naturally without medication?",
    "input": "",
    "output": "Establish a consistent sleep routine. Reduce screens 1 hour before bed. Do Yoga Nidra. Sleep in a dark, cool room. Drink ashwagandha milk. Brahmi oil head massage before bedtime is a classical sleep remedy."
  },
  {
    "instruction": "What foods cause sleep disturbance and should be avoided at night?",
    "input": "",
    "output": "Avoid caffeine after 2 PM. Avoid spicy or heavy fried foods at dinner. Alcohol disrupts deep sleep cycles. Sugary foods cause blood sugar spikes that disturb sleep."
  },
  {
    "instruction": "How to use brahmi to reduce mental stress?",
    "input": "",
    "output": "Take brahmi powder (1/4 tsp) in warm milk or water. Brahmi oil head massage is deeply calming - warm a few drops and massage into the scalp. It is safe for long-term daily use."
  },
  {
    "instruction": "What is the best yoga for reducing daily stress?",
    "input": "",
    "output": "Restorative yoga poses like Child Pose and Legs-up-the-Wall are excellent stress relievers. Surya Namaskar in the morning reduces cortisol. Pranayama daily is proven to reduce anxiety."
  },
  {
    "instruction": "How to help a child with irrational fears and sleep anxiety?",
    "input": "",
    "output": "Create a calming bedtime ritual with warm milk with a tiny pinch of nutmeg, a short story, and dim lighting. A lavender sachet near the bed helps. Consult a psychologist if anxiety is persistent."
  },
  {
    "instruction": "I have a sudden toothache. What home remedy helps?",
    "input": "",
    "output": "Place a whole clove or apply a tiny drop of clove oil directly on the cavity for temporary pain relief. Gargle with warm salt water to reduce inflammation. See a dentist as soon as possible."
  },
  {
    "instruction": "What helps with swollen gums at home?",
    "input": "",
    "output": "Gargle with warm salt water twice a day. Apply clove oil gently on the swollen gum. Neem twig chewing or neem water gargle is an ancient remedy for gum disease."
  },
  {
    "instruction": "How to whiten teeth naturally at home?",
    "input": "",
    "output": "Oil pulling with 1 tbsp of sesame or coconut oil for 10-15 minutes removes surface stains. A banana peel rubbed on teeth for 2 minutes is a gentle whitening method. Always brush with fluoride toothpaste afterward."
  },
  {
    "instruction": "What reduces toothache pain quickly?",
    "input": "",
    "output": "Apply a small piece of garlic directly to the affected tooth - allicin in garlic has numbing and antibacterial properties. Clove oil on a cotton ball is the fastest traditional pain relief method."
  },
  {
    "instruction": "How to prevent tooth decay with Ayurvedic methods?",
    "input": "",
    "output": "Practice oil pulling with sesame oil for 10 minutes daily. Use a neem toothbrush. Rinse mouth with triphala powder water after meals. Avoid refined sugars and sticky sweets."
  },
  {
    "instruction": "What helps a child with teething pain?",
    "input": "",
    "output": "Gently rub the gums with a clean finger or a cold wet cloth. A chilled teething ring helps. Clove oil diluted significantly in coconut oil can be applied sparingly for older infants - consult your pediatrician first."
  },
  {
    "instruction": "How to deal with wisdom tooth pain at home?",
    "input": "",
    "output": "Rinse with warm salt water several times a day. Apply clove oil on the affected area. An ice pack on the jaw reduces swelling. Wisdom tooth pain requires dental evaluation; home remedies provide only temporary relief."
  },
  {
    "instruction": "What is oil pulling and how does it help oral health?",
    "input": "",
    "output": "Oil pulling involves swishing 1 tbsp of sesame or coconut oil in your mouth for 10-15 minutes each morning before eating. It reduces harmful oral bacteria, whitens teeth, and reduces gum inflammation."
  },
  {
    "instruction": "How to handle bleeding gums at home?",
    "input": "",
    "output": "Rinse your mouth with warm salt water. Apply a small amount of aloe vera gel on the gums. Eat vitamin C-rich foods as deficiency is a common cause of bleeding gums. See a dentist if bleeding is persistent."
  },
  {
    "instruction": "What foods help maintain strong teeth naturally?",
    "input": "",
    "output": "Eat calcium-rich foods: dairy, sesame seeds, ragi (finger millet), and leafy greens. Crunchy vegetables like carrots act as natural tooth brushers. Avoid sugary and acidic foods."
  }
]

print(f'Seed samples: {len(SEED_DATA)}')
seed_path = DATA_RAW / 'seed_data.jsonl'
with open(seed_path, 'w', encoding='utf-8') as fh:
    for item in SEED_DATA:
        fh.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f'Saved -> {seed_path}')


In [ ]:
# Cell 5 - Data cleaning and validation
import re, json
from pathlib import Path

def clean_text(t):
    return re.sub(r'\s+', ' ', str(t).strip())

def validate(s):
    return (len(s.get('instruction','').strip()) >= 10
            and len(s.get('output','').strip()) >= 30)

def clean_dataset(samples):
    cleaned, seen = [], set()
    for s in samples:
        s = {k: clean_text(v) for k, v in s.items()}
        key = s['instruction'].lower()[:60]
        if key in seen:
            continue
        seen.add(key)
        if validate(s):
            cleaned.append(s)
    return cleaned

cleaned_data = clean_dataset(SEED_DATA)
print(f'Original       : {len(SEED_DATA)}')
print(f'After cleaning : {len(cleaned_data)}')

DATA_RAW   = Path('.').resolve() / 'data' / 'raw'
clean_path = DATA_RAW / 'seed_data_clean.jsonl'
with open(clean_path, 'w', encoding='utf-8') as fh:
    for item in cleaned_data:
        fh.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f'Cleaned data saved -> {clean_path}')


In [ ]:
# Cell 6 - Train/Val/Test split and Alpaca prompt formatting
# Prompt format: ### Instruction / ### Input / ### Response
import random, json
from pathlib import Path

DATA_PROC = Path('.').resolve() / 'data' / 'processed'
DATA_PROC.mkdir(parents=True, exist_ok=True)

DISCLAIMER = (
    '\n\nWARNING: This is a traditional home remedy suggestion only. '
    'Please consult a licensed physician for serious or persistent health conditions.'
)

H_INSTR = '### Instruction:'
H_INPUT = '### Input:'
H_RESP  = '### Response:'

def format_alpaca(sample):
    instr = sample['instruction']
    inp   = sample.get('input', '').strip()
    out   = sample['output']
    text  = H_INSTR + '\n' + instr + '\n\n'
    if inp:
        text += H_INPUT + '\n' + inp + '\n\n'
    text += H_RESP + '\n' + out
    return {'text': text}

data = [format_alpaca(s) for s in cleaned_data]
random.seed(42)
random.shuffle(data)
n = len(data)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
splits  = {
    'train': data[:n_train],
    'val'  : data[n_train:n_train + n_val],
    'test' : data[n_train + n_val:],
}
for name, split in splits.items():
    path = DATA_PROC / f'{name}.jsonl'
    with open(path, 'w', encoding='utf-8') as fh:
        for item in split:
            fh.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f'{name:5s}: {len(split):3d} samples -> {path}')

print('\nPrompt preview:')
print(data[0]['text'][:350] + '...')


In [ ]:
# Cell 7 - Load TinyLlama-1.1B-Chat with 4-bit quantisation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

if USE_GPU:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map='cpu',
        trust_remote_code=True,
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters  : {model.num_parameters() / 1e9:.2f}B')


In [ ]:
# Cell 8 - Apply LoRA (Parameter Efficient Fine-Tuning)
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'v_proj',
        'k_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5-1% trainable params - that is correct for LoRA r=16


In [ ]:
# Cell 9 - Fine-tune with SFTTrainer
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import wandb, json
from pathlib import Path

DATA_PROC  = Path('.').resolve() / 'data' / 'processed'
OUTPUT_DIR = Path('.').resolve() / 'outputs' / 'remedies-slm'

def load_jsonl(p):
    with open(p, encoding='utf-8') as fh:
        return [json.loads(l) for l in fh]

train_ds = Dataset.from_list(load_jsonl(DATA_PROC / 'train.jsonl'))
val_ds   = Dataset.from_list(load_jsonl(DATA_PROC / 'val.jsonl'))

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=20,
    save_steps=50,
    save_total_limit=3,
    fp16=USE_GPU,
    report_to='wandb',
    run_name=WANDB_RUN_NAME,
    max_seq_length=512,
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)

print('Starting training...')
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
print(f'Model adapter saved to {OUTPUT_DIR}')
wandb.finish()


In [ ]:
# Cell 10 - Inference test (10 sample prompts)
from transformers import pipeline
from pathlib import Path

OUTPUT_DIR = Path('.').resolve() / 'outputs' / 'remedies-slm'

pipe = pipeline(
    'text-generation',
    model=str(OUTPUT_DIR),
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
)

H_RESP = '### Response:'

TEST_PROMPTS = [
    'What home remedy can I use for a headache?',
    'How to treat acidity naturally?',
    'My child has fever. What Indian remedy helps?',
    'What helps with bloating after meals?',
    'How do I relieve a toothache at home?',
    'Suggest a remedy for dry cough.',
    'What helps with joint pain and arthritis naturally?',
    'How to sleep better using home remedies?',
    'What Indian herb boosts immunity?',
    'How to treat a sunburn at home?',
]

DISCLAIMER = (
    'WARNING: Traditional home remedy suggestion only. '
    'Consult a doctor for serious conditions.'
)

print('=' * 60)
for i, symptom in enumerate(TEST_PROMPTS, 1):
    prompt = '### Instruction:\n' + symptom + '\n\n### Response:\n'
    result = pipe(prompt)[0]['generated_text']
    answer = result.split(H_RESP)[-1].strip().split('###')[0].strip()
    print(f'[{i}] Q: {symptom}')
    print(f'    A: {answer[:200]}')
    print(f'    -> {DISCLAIMER}')
    print('-' * 60)


In [ ]:
# Cell 11 - ROUGE-L and BLEU evaluation on test split
import json, evaluate
from pathlib import Path

rouge_metric = evaluate.load('rouge')
bleu_metric  = evaluate.load('bleu')

DATA_PROC  = Path('.').resolve() / 'data' / 'processed'
test_data  = [json.loads(l) for l in open(DATA_PROC / 'test.jsonl', encoding='utf-8')]

predictions, references = [], []
H_RESP = '### Response:'

for sample in test_data:
    parts   = sample['text'].split(H_RESP)
    prompt  = parts[0] + H_RESP + '\n'
    ref_ans = parts[1].strip()
    gen     = pipe(prompt, max_new_tokens=200)[0]['generated_text']
    pred    = gen.split(H_RESP)[-1].strip().split('###')[0].strip()
    predictions.append(pred)
    references.append(ref_ans)

rouge_scores = rouge_metric.compute(predictions=predictions, references=references)
bleu_scores  = bleu_metric.compute(predictions=predictions,
                                    references=[[r] for r in references])

print('=' * 40)
print(f'ROUGE-1  : {rouge_scores["rouge1"]:.4f}   (target > 0.35)')
print(f'ROUGE-L  : {rouge_scores["rougeL"]:.4f}   (target > 0.35)')
print(f'BLEU     : {bleu_scores["bleu"]:.4f}   (target > 0.20)')
print('=' * 40)


In [ ]:
# Cell 12 - Gradio demo UI
# Launches at http://localhost:7860
import gradio as gr
from pathlib import Path

OUTPUT_DIR = Path('.').resolve() / 'outputs' / 'remedies-slm'
H_RESP = '### Response:'

DISCLAIMER = (
    'WARNING: This is a traditional home remedy suggestion only. '
    'Please consult a licensed physician for serious or persistent health conditions. '
    'This model may make mistakes.'
)

def get_remedy(symptom: str) -> str:
    if not symptom.strip():
        return 'Please describe your symptom or health question.'
    prompt = '### Instruction:\n' + symptom.strip() + '\n\n### Response:\n'
    result = pipe(prompt, max_new_tokens=250)[0]['generated_text']
    answer = result.split(H_RESP)[-1].strip().split('###')[0].strip()
    return answer + '\n\n' + DISCLAIMER

with gr.Blocks(title='India Home Remedies SLM', theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        '# India Home Remedies SLM\n'
        'Ask about any common health symptom and get traditional Indian home remedy suggestions.\n\n'
        '> **Disclaimer**: For informational purposes only. Consult a doctor for serious conditions.'
    )
    with gr.Row():
        with gr.Column():
            symptom_input = gr.Textbox(
                label='Describe your symptom or health concern',
                placeholder='e.g. I have a headache and feel tired',
                lines=3,
            )
            submit_btn = gr.Button('Get Remedy', variant='primary')
        with gr.Column():
            remedy_output = gr.Textbox(label='Suggested Home Remedy', lines=8)
    gr.Examples(
        examples=[
            ['I have a bad cold and blocked nose'],
            ['Feeling very bloated after dinner'],
            ['Mild fever since morning'],
            ['Toothache on the left side'],
            ['Cannot sleep well at night'],
        ],
        inputs=symptom_input,
    )
    submit_btn.click(fn=get_remedy, inputs=symptom_input, outputs=remedy_output)

demo.launch(share=False, server_port=7860)
print('Gradio UI running at http://localhost:7860')


In [ ]:
# Cell 13 - Write FastAPI server code (api.py)
# Run separately: uvicorn api:app --host 0.0.0.0 --port 8000
from pathlib import Path

API_LINES = [
    'from fastapi import FastAPI\n',
    'from pydantic import BaseModel\n',
    'from pathlib import Path\n',
    'from transformers import pipeline\n\n',
    'app = FastAPI(title="India Home Remedies SLM API")\n\n',
    'OUTPUT_DIR = Path(".").resolve() / "outputs" / "remedies-slm"\n',
    'pipe = pipeline("text-generation", model=str(OUTPUT_DIR), max_new_tokens=200)\n\n',
    'DISCLAIMER = (\n',
    '    "WARNING: Traditional home remedy suggestion only. "\n',
    '    "Consult a licensed physician for serious conditions."\n',
    ')\n\n',
    'class SymptomRequest(BaseModel):\n',
    '    symptom: str\n\n',
    '@app.post("/remedy")\n',
    'def get_remedy(req: SymptomRequest):\n',
    '    prompt = "### Instruction:\\n" + req.symptom + "\\n\\n### Response:\\n"\n',
    '    result = pipe(prompt)[0]["generated_text"]\n',
    '    answer = result.split("### Response:")[-1].strip().split("###")[0].strip()\n',
    '    return {"symptom": req.symptom, "remedy": answer, "disclaimer": DISCLAIMER}\n\n',
    '@app.get("/health")\n',
    'def health():\n',
    '    return {"status": "ok"}\n',
]

api_path = Path('.').resolve() / 'api.py'
with open(api_path, 'w', encoding='utf-8') as fh:
    fh.writelines(API_LINES)
print(f'FastAPI code written to {api_path}')
print('Run with: uvicorn api:app --host 0.0.0.0 --port 8000')
print('Test with: curl -X POST http://localhost:8000/remedy -H "Content-Type: application/json" -d "{\"symptom\":\"headache\"}"')


## Next Steps

1. **Expand dataset** - Add more Q&A pairs (target 2000-5000 samples)
2. **Tune hyperparameters** - Try LoRA r=32 or more epochs if underfitting
3. **CPU export** - Convert to GGUF for llama.cpp lightweight CPU deployment
4. **HuggingFace Hub** - Push adapter and tokenizer for sharing

---
*Built to preserve and share India's traditional healing wisdom.*
